# Model used for prediction

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import json
import os

In [2]:
# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Enable memory growth to prevent TensorFlow from allocating all GPU memory
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"✓ {len(gpus)} Physical GPU(s), {len(logical_gpus)} Logical GPU(s) available")
        print(f"GPU Name: {gpus[0].name}")
        
        # Enable mixed precision for RTX GPUs (faster training)
        from tensorflow.keras import mixed_precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print("✓ Mixed precision enabled (FP16) for faster training on RTX GPU")
        
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("⚠ No GPU detected. Training will use CPU (much slower)")
    print("To enable GPU, install CUDA and cuDNN compatible with TensorFlow")

print("=" * 60)
print()


⚠ No GPU detected. Training will use CPU (much slower)
To enable GPU, install CUDA and cuDNN compatible with TensorFlow



In [3]:
# Set parameters
IMG_SIZE = 227  # AlexNet input size
BATCH_SIZE = 32
EPOCHS = 50
DATA_DIR = 'img_dataset'

In [4]:
# Data augmentation and preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)


In [5]:
# Load training data
train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 3832 images belonging to 4 classes.


In [6]:
# Load validation data
validation_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 955 images belonging to 4 classes.


In [7]:
# Get class labels
class_labels = {v: k for k, v in train_generator.class_indices.items()}
num_classes = len(class_labels)

In [8]:
print(f"Found {num_classes} classes: {list(class_labels.values())}")

Found 4 classes: ['Corn__common_rust', 'Corn__gray_leaf_spot', 'Corn__healthy', 'Corn__northern_leaf_blight']


In [9]:
# Build AlexNet architecture
def create_alexnet(input_shape, num_classes):
    model = models.Sequential([
        # Conv Layer 1
        layers.Conv2D(96, kernel_size=11, strides=4, activation='relu', 
                     input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=3, strides=2),
        
        # Conv Layer 2
        layers.Conv2D(256, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=3, strides=2),
        
        # Conv Layer 3
        layers.Conv2D(384, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        
        # Conv Layer 4
        layers.Conv2D(384, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        
        # Conv Layer 5
        layers.Conv2D(256, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=3, strides=2),
        
        # Flatten and FC layers
        layers.Flatten(),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model


In [10]:
# Create model
model = create_alexnet((IMG_SIZE, IMG_SIZE, 3), num_classes)

c:\Users\Sajal\OneDrive_not\Major_Project\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
# Display model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 57, 57, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 57, 57, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 28, 28, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 13, 13, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 13, 13, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 13, 13, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │        16,388 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,303,236 (222.41 MB)

 Trainable params: 58,300,484 (222.40 MB)

 Non-trainable params: 2,752 (10.75 KB)

In [13]:
# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    )
]

In [ ]:
# Train model
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 102s 824ms/step - accuracy: 0.7685 - loss: 0.9176 - val_accuracy: 0.2283 - val_loss: 2.8140 - learning_rate: 1.0000e-04
Epoch 2/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 101s 843ms/step - accuracy: 0.8450 - loss: 0.4927 - val_accuracy: 0.2555 - val_loss: 4.6558 - learning_rate: 1.0000e-04
Epoch 3/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 96s 796ms/step - accuracy: 0.8737 - loss: 0.3790 - val_accuracy: 0.3466 - val_loss: 2.6763 - learning_rate: 1.0000e-04
Epoch 4/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 103s 862ms/step - accuracy: 0.8779 - loss: 0.3449 - val_accuracy: 0.5738 - val_loss: 1.7789 - learning_rate: 1.0000e-04
Epoch 5/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 111s 924ms/step - accuracy: 0.8938 - loss: 0.2919 - val_accuracy: 0.6283 - val_loss: 2.2538 - learning_rate: 1.0000e-04
Epoch 6/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 107s 889ms/step - accuracy: 0.9011 - loss: 0.2772 - val_accuracy: 0.5927 - val_loss: 4.2368 - learning_rate: 1.0000e-04
Epoch 7/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 1